# Detector de Contradicciones — 23F
**Pipeline:** Extracción de statements → Embeddings multilingüe → Candidatos similares → NLI → Reporte

---
## 0 · Configuración centralizada
Cambiamos rutas y parámetros solo en `config.json`.

In [1]:
import json
import os

# ── Carga config.json (debe estar en la misma carpeta que este notebook) ──
CONFIG_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'config.json')

with open(CONFIG_PATH, encoding='utf-8') as f:
    cfg = json.load(f)

# Rutas
RUTA_CSV          = cfg['rutas']['csv_principal']
RUTA_STATEMENTS   = cfg['rutas']['output_statements']
RUTA_EMBEDDINGS   = cfg['rutas']['output_embeddings']
RUTA_CONTRA       = cfg['rutas']['output_contradicciones']

# Modelos
MODELO_EMBEDDINGS = cfg['modelos']['embeddings']
MODELO_NLI        = cfg['modelos']['nli']

# Parámetros
SIM_MIN           = cfg['parametros']['similitud_minima']
CONFIANZA_MIN     = cfg['parametros']['confianza_contradiccion_minima']
MIN_PALABRAS      = cfg['parametros']['min_palabras_statement']
MAX_VECINOS       = cfg['parametros']['max_vecinos_nli']

# Ajuste práctico tras diagnóstico: en este dataset las similitudes máximas
# entre documentos distintos rondan 0.69, así que 0.75 deja df_pares vacío.
# Si config.json trae un umbral demasiado alto, lo bajamos automáticamente.
if SIM_MIN >= 0.70:
    print(f'⚠️ SIM_MIN={SIM_MIN} es demasiado alto para este dataset. Se ajusta a 0.55.')
    SIM_MIN = 0.55

# Crear carpeta outputs si no existe
os.makedirs('outputs', exist_ok=True)

print('✅ Config cargada correctamente')
print(f'   CSV:      {RUTA_CSV}')
print(f'   Modelo embeddings: {MODELO_EMBEDDINGS}')
print(f'   Modelo NLI:        {MODELO_NLI}')
print(f'   Similitud mínima:  {SIM_MIN} | Confianza NLI mínima: {CONFIANZA_MIN}')


✅ Config cargada correctamente
   CSV:      23f_scrappedDF.csv
   Modelo embeddings: paraphrase-multilingual-mpnet-base-v2
   Modelo NLI:        MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
   Similitud mínima:  0.55 | Confianza NLI mínima: 0.55


---
## 1 · Carga de datos

In [2]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv(RUTA_CSV)
print(f'Documentos cargados: {len(df)}')
print(f'Columnas: {df.columns.tolist()}')
df.head(3)

Documentos cargados: 167
Columnas: ['title', 'url', 'pages', 'summary', 'tags', 'transcript', 'filename']


,title,url,pages,summary,tags,transcript,filename
0,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1860,3,El juicio oral 2/81 celebrado en febrero de 19...,No consta||Luis Arana Lorite:Teniente Coronel|...,C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\...,Vista oral 281 del Consejo Supremo de Justicia...
1,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1859,4,Resumen global del documento:\n\nEl documento ...,Ministerio Fiscal||PRESIDENTE DEL CONSEJO||Con...,C/SG/2896/22-02-82\n\n# NOTA INFORMATIVA\n\nAS...,Vista oral 281 del Consejo Supremo de Justicia...
2,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1858,5,Resumen global del documento:\n\nEl documento ...,No consta||Tribunal||Defensores||Director de D...,C/SG/2992/24-02-82\n\n# NOTA INFORMATIVA\n\nAS...,Vista oral 281 del Consejo Supremo de Justicia...


---
## 2 · Extracción de fecha y limpieza

In [4]:
MESES = {
    'enero': '01', 'febrero': '02', 'marzo': '03', 'abril': '04',
    'mayo': '05', 'junio': '06', 'julio': '07', 'agosto': '08',
    'septiembre': '09', 'octubre': '10', 'noviembre': '11', 'diciembre': '12'
}

def extraer_fecha(titulo):
    """Extrae fecha del título en formato YYYY-MM-DD."""
    m = re.search(r'(\d{1,2}) de (\w+) de (\d{4})', str(titulo), re.IGNORECASE)
    if m:
        dia, mes_txt, anio = m.groups()
        mes = MESES.get(mes_txt.lower(), '00')
        return f'{anio}-{mes}-{int(dia):02d}'
    return None

def limpiar_texto(text):
    """Normaliza espacios y saltos de línea."""
    if pd.isna(text):
        return ''
    text = text.replace('\r', '\n')
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n+', '\n', text)
    return text.strip()

df['fecha']      = df['title'].apply(extraer_fecha)
df['transcript'] = df['transcript'].apply(limpiar_texto)

print(f'Documentos con fecha detectada: {df["fecha"].notna().sum()} / {len(df)}')
print('Rango temporal:', df['fecha'].dropna().min(), '→', df['fecha'].dropna().max())
df[['title', 'fecha']].head()

Documentos con fecha detectada: 83 / 167
Rango temporal: 1981-02-10 → 1987-10-19


,title,fecha
0,Vista oral 2/81 del Consejo Supremo de Justici...,1982-02-20
1,Vista oral 2/81 del Consejo Supremo de Justici...,1982-02-22
2,Vista oral 2/81 del Consejo Supremo de Justici...,1982-02-24
3,Vista oral 2/81 del Consejo Supremo de Justici...,1982-02-25
4,Vista oral 2/81 del Consejo Supremo de Justici...,1982-02-26


---
## 3 · Extracción de declarantes y statements

**tres niveles de captura**
1. Líneas con verbos declarativos («niega que», «afirma que»…)
2. Bullets informativos tras encabezados de declarante
3. Frases largas del transcript general (fallback)

Además normalizamos los nombres de declarantes para poder agrupar por persona.

In [17]:
# ── Patrones de encabezado de declarante ──────────────────────────────────
PATRON_DECLARANTE = re.compile(
    r'(?:declaraci[oó]n(?:es)?|interrogatorio|careo)\s+(?:del?|a\s+l?[ao]s?|entre)?\s*'
    r'(?:(?:Teniente\s+(?:General|Coronel)|General|Coronel|Comandante|Capitán|'
    r'Capitán\s+General|Almirante|Brigada|Sargento|Mayor)\s+)?'
    r'(?:D\.?\s*)?([A-ZÁÉÍÓÚÜÑ][A-ZÁÉÍÓÚÜÑa-záéíóúüñ\s]{3,50})',
    re.IGNORECASE
)

# También capturamos líneas tipo "Interrogatorio al Gral. Armada"
PATRON_DECLARANTE2 = re.compile(
    r'(?:Interrogatorio|Declaraci[oó]n)\s+(?:al?|del?)\s+'
    r'(?:(?:Teniente\s+)?(?:General|Coronel|Comandante|Capitán)\s+)?'
    r'([A-ZÁÉÍÓÚÜÑ][A-ZÁÉÍÓÚÜÑa-záéíóúüñ\s]{3,50})',
    re.IGNORECASE
)

# Verbos declarativos (indican que la línea tiene contenido semántico relevante)
VERBOS_DECLARATIVOS = [
    'afirma que', 'afirmó que', 'manifiesta que', 'manifestó que',
    'señala que', 'señaló que', 'dice que', 'dijo que',
    'niega que', 'negó que', 'niega haber', 'negó haber',
    'sostiene que', 'sostuvo que', 'indica que', 'indicó que',
    'explica que', 'explicó que', 'reconoce que', 'reconoció que',
    'admite que', 'admitió que', 'asegura que', 'aseguró que',
    'declara que', 'declaró que', 'alega que', 'alegó que',
    'insiste en que', 'insistió en que', 'asevera que',
    'puede resumirse en', 'pueden resumirse en'
]

def normalizar_declarante(nombre):
    """Limpia tratamientos y mayúsculas para poder agrupar por persona."""
    if not nombre:
        return None
    nombre = nombre.upper().strip()
    for prefijo in [r'\bD\.\s*', r'\bDON\b\s*', r'\bDOÑA\b\s*',
                    r'\bTNTE\.?\s*CNEL\.?\s*', r'\bTENIENTE\s+CORONEL\s+',
                    r'\bTENIENTE\s+GENERAL\s+', r'\bGENERAL\s+',
                    r'\bCORONEL\s+', r'\bCOMANDANTE\s+', r'\bCAPITÁN\s+',
                    r'\bALMIRANTE\s+']:
        nombre = re.sub(prefijo, '', nombre, flags=re.IGNORECASE)
    nombre = re.sub(r'\s+', ' ', nombre).strip()
    # Quitar apellidos muy genéricos que no identifican a nadie
    if len(nombre) < 4:
        return None
    return nombre

def es_linea_statement(linea):
    """True si la línea contiene una declaración sustantiva."""
    low = linea.lower()
    if any(v in low for v in VERBOS_DECLARATIVOS):
        return True
    # Bullet con suficiente contenido
    if linea.startswith('-') and len(linea.split()) >= MIN_PALABRAS:
        return True
    return False

def extraer_declarante_linea(linea):
    """Intenta extraer el nombre del declarante de una línea de encabezado."""
    for pat in [PATRON_DECLARANTE, PATRON_DECLARANTE2]:
        m = pat.search(linea)
        if m:
            return m.group(1).strip()
    return None

print('✅ Funciones de extracción definidas')

✅ Funciones de extracción definidas


In [18]:
rows = []

for doc_idx, doc in df.iterrows():
    texto    = doc['transcript']
    titulo   = doc['title']
    fecha    = doc['fecha']
    filename = doc['filename']
    lineas   = texto.split('\n')

    declarante_actual = None
    i = 0

    while i < len(lineas):
        linea = lineas[i].strip()
        if not linea:
            i += 1
            continue

        # ¿Es un encabezado de declarante?
        nuevo_declarante = extraer_declarante_linea(linea)
        if nuevo_declarante:
            declarante_actual = normalizar_declarante(nuevo_declarante)
            i += 1
            continue

        # ¿Es una línea con contenido declarativo?
        if declarante_actual and es_linea_statement(linea):
            # Limpia el bullet inicial
            texto_limpio = re.sub(r'^[-–•.\s]+', '', linea).strip()
            if len(texto_limpio.split()) >= MIN_PALABRAS:
                rows.append({
                    'doc_idx'    : doc_idx,
                    'titulo'     : titulo,
                    'fecha'      : fecha,
                    'filename'   : filename,
                    'declarante' : declarante_actual,
                    'statement'  : texto_limpio
                })
        i += 1

df_st = pd.DataFrame(rows)

print(f'Statements extraídos:     {len(df_st)}')
print(f'Declarantes únicos:       {df_st["declarante"].nunique()}')
print(f'Documentos con al menos 1 statement: {df_st["doc_idx"].nunique()}')
print()
print('Top 15 declarantes por nº de statements:')
print(df_st['declarante'].value_counts().head(15))

Statements extraídos:     1462
Declarantes únicos:       196
Documentos con al menos 1 statement: 50

Top 15 declarantes por nº de statements:
declarante
QUE HA HECHO                                           66
L SR                                                   37
POLÍTICAS Y TECNOLÓGICAS SE ENCUENTRAN EN TEXTO IMP    34
SUYA ANTERIOR CON LAS DE OTROS                         33
TENGA TAMPOCO NADA RELEVANTE                           29
CERTIFICADAS QUE FIGURAN EN EL PROCEDIMIENTO           28
FISCAL AL TCOL                                         26
AL TENIENTE DE LA GUARDIA CIVIL ALONSO HERNAIZ         26
FISCAL AL IBAÑEZ INGLÉS                                26
AL TENIENTE DE LA GUARDIA CIVIL BOZA CARRANCO          25
LA QUE PUEDE DESTACARSE LO SIGUIENTE                   24
L TTE                                                  23
PROCESADO IGNACIO ROMÁN                                22
TCOL D                                                 21
PROCESADO DE INFANTERÍA PASCUAL GA

In [19]:
# Guardamos los statements para inspeccionarlos
df_st.to_csv(RUTA_STATEMENTS, index=False, encoding='utf-8-sig')
print(f'✅ Statements guardados en: {RUTA_STATEMENTS}')
df_st.head()

✅ Statements guardados en: outputs/statements_extraidos.csv


,doc_idx,titulo,fecha,filename,declarante,statement
0,0,Vista oral 2/81 del Consejo Supremo de Justici...,1982-02-20,Vista oral 281 del Consejo Supremo de Justicia...,TEJERO,"Descanso de 11,50 horas a 12,13 horas."
1,0,Vista oral 2/81 del Consejo Supremo de Justici...,1982-02-20,Vista oral 281 del Consejo Supremo de Justicia...,PARCIALES,Uno de los defensores manifiesta que habían ll...
2,0,Vista oral 2/81 del Consejo Supremo de Justici...,1982-02-20,Vista oral 281 del Consejo Supremo de Justicia...,PARCIALES,Reticencias y sonrisas de defensores y codefen...
3,0,Vista oral 2/81 del Consejo Supremo de Justici...,1982-02-20,Vista oral 281 del Consejo Supremo de Justicia...,ALGUNOS PROCESADOS HACE QUE EL NOMBRE DE SU MA...,El CESID permanece en primera fila de declarac...
4,0,Vista oral 2/81 del Consejo Supremo de Justici...,1982-02-20,Vista oral 281 del Consejo Supremo de Justicia...,TEJERO AFIRMA QUE HABÍA MÁS PERSONAS DEL CESID...,Los familiares ván tomando más actividad y se ...


---
## 4 · Embeddings multilingüe

Usamos `paraphrase-multilingual-mpnet-base-v2`: entrenado en 50+ idiomas, mucho mejor para español que `all-MiniLM-L6-v2` (que era solo inglés).

> **Primera vez:** descargará ~1 GB del modelo. Se cachea automáticamente.

In [8]:
from sentence_transformers import SentenceTransformer

print(f'Cargando modelo: {MODELO_EMBEDDINGS} ...')
model_emb = SentenceTransformer(MODELO_EMBEDDINGS)
print('✅ Modelo cargado')

textos = df_st['statement'].tolist()
print(f'Generando embeddings para {len(textos)} statements...')

embeddings = model_emb.encode(
    textos,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f'✅ Shape de embeddings: {embeddings.shape}')

Cargando modelo: paraphrase-multilingual-mpnet-base-v2 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modelo cargado
Generando embeddings para 1462 statements...


Batches:   0%|          | 0/23 [00:00<?, ?it/s]

✅ Shape de embeddings: (1462, 768)


In [9]:
# Guardamos embeddings para no recalcular (pueden tardar varios minutos)
df_emb = df_st.copy()
df_emb['embedding'] = [','.join(map(str, e)) for e in embeddings]
df_emb.to_csv(RUTA_EMBEDDINGS, index=False, encoding='utf-8-sig')
print(f'✅ Embeddings guardados en: {RUTA_EMBEDDINGS}')

✅ Embeddings guardados en: outputs/embeddings.csv


---
## 5 · Búsqueda de pares candidatos

**Clave de este paso:** solo comparamos statements **del mismo declarante** en **documentos distintos**.
- Evita falsos positivos de comparar personas diferentes.
- Busca exactamente lo que queremos: ¿dijo lo mismo en distintas sesiones?

Usamos `NearestNeighbors` con métrica coseno para encontrar los `k` más similares eficientemente.

In [10]:
from sklearn.neighbors import NearestNeighbors
import pandas as pd

pares_candidatos = []
sims_entre_docs = []
total_comparaciones = 0
saltados_mismo_doc = 0

declarantes_validos = df_st['declarante'].value_counts()
# Solo declarantes con >= 2 statements: necesitamos al menos 2 para comparar.
declarantes_validos = declarantes_validos[declarantes_validos >= 2].index.tolist()

print(f'Declarantes con >= 2 statements: {len(declarantes_validos)}')
print(f'SIM_MIN usado: {SIM_MIN}')
print()

for declarante in declarantes_validos:
    mask = df_st['declarante'] == declarante
    idx_sub = df_st[mask].index.tolist()
    emb_sub = embeddings[mask]
    docs_sub = df_st.loc[mask, 'doc_idx'].values

    if len(idx_sub) < 2:
        continue

    k = min(MAX_VECINOS + 1, len(idx_sub))
    nn = NearestNeighbors(
        n_neighbors=k,
        metric='cosine',
        algorithm='brute'
    )
    nn.fit(emb_sub)
    distancias, vecinos = nn.kneighbors(emb_sub)

    for local_i in range(len(idx_sub)):
        for rank in range(1, k):
            local_j = vecinos[local_i][rank]
            global_i = idx_sub[local_i]
            global_j = idx_sub[local_j]

            if global_i >= global_j:
                continue  # Evita duplicados: A-B y B-A.

            total_comparaciones += 1

            # Solo comparamos statements del mismo declarante, pero en documentos distintos.
            if docs_sub[local_i] == docs_sub[local_j]:
                saltados_mismo_doc += 1
                continue

            sim = 1 - distancias[local_i][rank]
            sims_entre_docs.append(float(sim))

            if sim >= SIM_MIN:
                pares_candidatos.append({
                    'i': global_i,
                    'j': global_j,
                    'declarante': declarante,
                    'similitud': round(float(sim), 4)
                })

# Importante: aunque no haya filas, dejamos creadas las columnas para evitar KeyError.
df_pares = pd.DataFrame(
    pares_candidatos,
    columns=['i', 'j', 'declarante', 'similitud']
).drop_duplicates(subset=['i', 'j'])

print('Diagnóstico de búsqueda:')
print(f'   Comparaciones totales evaluadas:      {total_comparaciones}')
print(f'   Saltadas por mismo documento:         {saltados_mismo_doc}')
print(f'   Comparaciones entre documentos:       {len(sims_entre_docs)}')
print(f'✅ Pares candidatos encontrados: {len(df_pares)}')

if df_pares.empty:
    print('⚠️ No se encontraron pares candidatos.')
    if sims_entre_docs:
        s = pd.Series(sims_entre_docs)
        print('\nDistribución de similitudes entre documentos distintos:')
        print(s.describe())
        print('\nTop 20 similitudes observadas:')
        print(s.sort_values(ascending=False).head(20))
        print('\nSugerencia: baja SIM_MIN a 0.50 o 0.45 si quieres más recall.')
    else:
        print('No hubo comparaciones válidas entre documentos distintos.')
else:
    print('\nDistribución por declarante:')
    print(df_pares['declarante'].value_counts().head(10))
    print('\nTop pares candidatos:')
    display(df_pares.sort_values('similitud', ascending=False).head(10))



Declarantes con >= 2 statements: 161
SIM_MIN usado: 0.55

Diagnóstico de búsqueda:
   Comparaciones totales evaluadas:      5719
   Saltadas por mismo documento:         5392
   Comparaciones entre documentos:       327
✅ Pares candidatos encontrados: 34

Distribución por declarante:
declarante
L SR                17
L TTE                5
TCOL D               4
OTROS DEFENSORES     4
SU DEFENSOR          3
GRAL                 1
Name: count, dtype: int64

Top pares candidatos:


,i,j,declarante,similitud
22,926,970,TCOL D,0.6975
31,228,272,OTROS DEFENSORES,0.6886
27,178,218,SU DEFENSOR,0.6739
25,930,977,TCOL D,0.6491
19,134,1295,L TTE,0.6412
7,1051,1125,L SR,0.6201
8,1051,1117,L SR,0.6201
21,146,1296,L TTE,0.6189
23,928,970,TCOL D,0.6179
0,614,1052,L SR,0.6092


In [11]:
# Vista rápida de los pares candidatos generados
print(df_pares.shape)
display(df_pares.head(10))



(34, 4)


,i,j,declarante,similitud
0,614,1052,L SR,0.6092
1,614,1053,L SR,0.5534
2,1050,1124,L SR,0.5972
3,1050,1117,L SR,0.5844
4,1050,1125,L SR,0.5819
5,1050,1106,L SR,0.5734
6,1050,1107,L SR,0.5544
7,1051,1125,L SR,0.6201
8,1051,1117,L SR,0.6201
9,1051,1126,L SR,0.6034


---
## 6 · NLI — Detección de contradicción

Para cada par candidato (similitud temática alta) le preguntamos al modelo NLI:
*¿El statement A contradice al statement B?*

**Truco importante:** pasamos el par en **ambos sentidos** (A→B y B→A) y nos quedamos con el máximo score de contradicción. Esto reduce falsos negativos de asimetría del modelo.

> ⚠️ Este paso puede tardar. Si tienes muchos pares, empieza con `df_pares.head(200)` para probar.

In [12]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
dispositivo_str = 'GPU' if device == 0 else 'CPU (Mac)'
print(f'Dispositivo: {dispositivo_str}')
print(f'Cargando modelo NLI: {MODELO_NLI} ...')

nli = pipeline(
    'text-classification',
    model=MODELO_NLI,
    return_all_scores=True,
    device=device
)
print('✅ Modelo NLI listo')

Dispositivo: CPU (Mac)
Cargando modelo NLI: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli ...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modelo NLI listo


In [13]:
def score_nli(texto_a, texto_b):
    """
    Devuelve (contradiccion, entailment, neutral) como máximos
    del par en ambas direcciones.
    """
    def normalizar(resultado):
        return {x['label'].lower(): x['score'] for x in resultado[0]}

    r_ab = normalizar(nli({'text': texto_a, 'text_pair': texto_b}))
    r_ba = normalizar(nli({'text': texto_b, 'text_pair': texto_a}))

    contradiccion = max(r_ab.get('contradiction', 0), r_ba.get('contradiction', 0))
    entailment    = max(r_ab.get('entailment', 0),    r_ba.get('entailment', 0))
    neutral       = max(r_ab.get('neutral', 0),       r_ba.get('neutral', 0))
    return contradiccion, entailment, neutral


# ── Aplicar NLI a todos los pares candidatos ──────────────────────────────
# Descomenta la línea siguiente para hacer prueba rápida con los primeros 100 pares:
# df_pares_test = df_pares.head(100)
df_pares_test = df_pares  # ← todos los pares

resultados = []
total = len(df_pares_test)

for n, (_, fila) in enumerate(df_pares_test.iterrows()):
    if n % 50 == 0:
        print(f'  Procesando par {n}/{total}...')

    i, j       = int(fila['i']), int(fila['j'])
    st_a       = df_st.loc[i, 'statement']
    st_b       = df_st.loc[j, 'statement']
    fecha_a    = df_st.loc[i, 'fecha']
    fecha_b    = df_st.loc[j, 'fecha']
    titulo_a   = df_st.loc[i, 'titulo']
    titulo_b   = df_st.loc[j, 'titulo']

    contra, entail, neutral = score_nli(st_a, st_b)

    resultados.append({
        'declarante'            : fila['declarante'],
        'statement_a'           : st_a,
        'statement_b'           : st_b,
        'doc_a'                 : titulo_a,
        'doc_b'                 : titulo_b,
        'fecha_a'               : fecha_a,
        'fecha_b'               : fecha_b,
        'similitud_tematica'    : fila['similitud'],
        'score_contradiccion'   : round(contra, 4),
        'score_entailment'      : round(entail, 4),
        'score_neutral'         : round(neutral, 4)
    })

df_resultados = pd.DataFrame(resultados)
print(f'\n✅ NLI completado para {len(df_resultados)} pares')

  Procesando par 0/34...


KeyError: 0